# Sensitivity to instrument effects

> How much the predictions of the OSSL models change when one physical instrument effect is applied to OSSL spectra

In [ ]:
#| default_exp sensitivity

In [ ]:
#| export
import numpy as np, pandas as pd, torch
from soilspecdl.all import predict
from ossl_rt_transfer.pretrain import rt_wns, preprocess
from ossl_rt_transfer.instrument_model import InstrumentModel

/Users/franckalbinet/aai-ws/.venv/lib/python3.13/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


In [ ]:
from functools import partial
from fastcore.test import *
from sklearn.metrics import r2_score
from soilspecdl.all import SpectralCNN, get_splitter
from soilspecdata.datasets.ossl import load_ossl
from soilspecdata.datasets.ringtrial import load_ringtrial
from ossl_rt_transfer.data import rt_path, ossl_props, ossl_overlap, ossl_matches
from ossl_rt_transfer.pretrain import ossl_xy, load_pretrained, pretrain_history
from ossl_rt_transfer.instrument_model import fit_instrument, paired_spectra

The cells that read OSSL and the ring-trial data are marked `#| rtdata`. The outputs on this page come from a local run.

An augmentation engine for pretraining has to cover the instrument effects that the OSSL models react to most. This notebook applies each effect of `InstrumentModel` alone to OSSL validation spectra, at magnitudes that real instruments can have, and measures how much the predictions of the 80-epoch models of `01_pretrain` change. Two measured differences give a scale: the difference spectrum D between the KSSL ring-trial spectra and the OSSL spectra of the same soils, and the effects fitted for MSU, IAEA and OSU in `03_instrument_model`.

## Effects in physical units

`InstrumentModel` sets its shift in points and its filter width as a logarithm, which suits fitting. `perturb` sets the same effects in physical units, and applies only the effects whose arguments differ from their defaults:

In [ ]:
#| export
def perturb(
    X, # Absorbance spectra on `rt_wns`, one per row
    shift_cm=0., # Displacement of every band, in cm⁻¹, positive towards higher wavenumbers
    stretch=0., # Displacement of each band as a fraction of its wavenumber
    sigma_cm=0., # Standard deviation of a Gaussian broadening filter, in cm⁻¹
    gamma=1., # Contrast: absorbance raised to this power
    gain=None, # B-spline coefficients added to a gain of 1, as in `InstrumentModel`
    baseline=None, # B-spline coefficients of an additive baseline, in absorbance
)->np.ndarray: # Perturbed spectra
    "Apply the effects of `InstrumentModel` to `X`, with parameters in physical units"
    wn = rt_wns.astype(float); dw = wn[1] - wn[0]
    on = {'shift': shift_cm or stretch, 'broaden': sigma_cm, 'contrast': gamma != 1, 'gain': gain is not None, 'baseline': baseline is not None}
    m = InstrumentModel(wn, [e for e, v in on.items() if v])
    with torch.no_grad():
        m.shift.copy_(torch.tensor([-(shift_cm + stretch*wn.mean())/dw, -stretch]))
        if sigma_cm: m.log_sigma.fill_(np.log(sigma_cm/dw))
        m.log_gamma.fill_(np.log(gamma))
        if gain is not None: m.g.add_(torch.tensor(gain))
        if baseline is not None: m.b.copy_(torch.tensor(baseline))
        return m(torch.tensor(X, dtype=torch.float64)).numpy()

A band at 2000 cm⁻¹ moves 2 cm⁻¹ higher with a shift of 2 cm⁻¹, or with a stretch of 0.001. A Gaussian filter with σ = 6 cm⁻¹ widens a band of σ = 10 cm⁻¹ from a full width at half maximum (FWHM) of 23.5 cm⁻¹ to √(10² + 6²) × 2.355 = 27.5 cm⁻¹. Counting the points above half the maximum, 2 cm⁻¹ apart, gives 22 and 26 cm⁻¹:

In [ ]:
wn = rt_wns.astype(float)
band = np.exp(-0.5*((wn - 2000)/10)**2)[None]
peak = lambda x: wn[x.argmax()]
fwhm = lambda x: (x[0] > x.max()/2).sum()*2
test_eq(peak(perturb(band, shift_cm=2)), 2002)
test_eq(peak(perturb(band, stretch=1e-3)), 2002)
fwhm(band), fwhm(perturb(band, sigma_cm=6))

(np.int64(22), np.int64(26))

`sensitivity` predicts the spectra once as they are and once for each case. It gives the mean of the change in prediction (`bias`) and its RMS (`rms`):

In [ ]:
#| export
def sensitivity(
    model, # Pretrained model, such as from `load_pretrained`
    X, # Absorbance spectra on `rt_wns`, one per row
    cases, # Dict from a name to a function that turns `X` into perturbed spectra, such as a `partial` of `perturb`
)->pd.DataFrame: # One row per case
    "Mean and RMS of the change in the predictions of `model` when each function in `cases` is applied to `X`"
    f = lambda x: predict(model, preprocess().fit_transform(x), rt_wns).ravel()
    p0 = f(X)
    d = {k: f(fn(X)) - p0 for k, fn in cases.items()}
    return pd.DataFrame({k: {'bias': v.mean(), 'rms': np.sqrt((v**2).mean())} for k, v in d.items()}).T

In [ ]:
x = np.random.default_rng(0).uniform(0.5, 1.5, (8, len(rt_wns)))
res = sensitivity(SpectralCNN(), x, {'none': lambda X: X, 'shift': partial(perturb, shift_cm=2)})
test_eq(res.loc['none'].tolist(), [0, 0])
res

,bias,rms
none,0.000000,0.000000
shift,0.003212,0.004589


## OSSL validation spectra

Each model is tested on spectra from its OSSL validation set, the 10% of samples that `pretrain` held out. `get_splitter` with the seed of `pretrain` gives the same split, and the models reach the R² that pretraining recorded at its last epoch. The validation RMSE gives a scale for the changes below. For K, the predictions and their changes are in `log1p(K)`:

In [ ]:
#| rtdata
om = load_ossl().mir(650, 4000)
models = {p: load_pretrained(f'ossl_{p}_snv_ep80') for p in ossl_props}
def valid_set(p):
    X, y, _ = ossl_xy(om, p)
    vi = get_splitter(0.9, 0.1, seed=42)(X, y)[1]
    return X[vi], y[vi]
val = {p: valid_set(p) for p in ossl_props}
pv = {p: predict(models[p], preprocess().fit_transform(X), rt_wns).ravel() for p, (X, y) in val.items()}
scores = pd.DataFrame({p: {'R²': r2_score(y, pv[p]), 'recorded R²': pretrain_history(f'ossl_{p}_snv_ep80').r2_score.iloc[-1],
                           'RMSE': np.sqrt(((pv[p] - y)**2).mean())} for p, (X, y) in val.items()}).T
test_close(scores['R²'], scores['recorded R²'], eps=1e-3)
scores.round(3)

,R²,recorded R²,RMSE
clay_perc,0.924,0.924,4.739
pH_H20,0.931,0.931,0.346
potassium_cmolkg,0.822,0.822,0.161


The tests use 2,000 of these spectra for each property. `sweep` runs `sensitivity` for the three models:

In [ ]:
#| rtdata
rng = np.random.default_rng(0)
Xs = {p: X[rng.choice(len(X), 2000, replace=False)] for p, (X, y) in val.items()}
def sweep(cases): return pd.concat({p: sensitivity(models[p], Xs[p], cases) for p in ossl_props}, axis=1)

## Wavenumber shift

A shift moves every band by the same number of cm⁻¹. In `03_instrument_model`, the shift fitted for IAEA is 2.4 to 2.5 cm⁻¹ in size. The cases go from 0.5 to 4 cm⁻¹ in each direction:

In [ ]:
#| rtdata
sweep({f'{s:+g} cm⁻¹': partial(perturb, shift_cm=s) for s in [-4, -2, -1, -0.5, 0.5, 1, 2, 4]}).round(2)

clay_perc        pH_H20       potassium_cmolkg      
               bias    rms   bias   rms             bias   rms
-4 cm⁻¹       16.59  20.74   0.15  0.37             0.19  0.28
-2 cm⁻¹       10.90  13.15   0.08  0.21             0.18  0.23
-1 cm⁻¹        4.32   5.37   0.04  0.11             0.11  0.13
-0.5 cm⁻¹      1.78   2.31   0.02  0.05             0.05  0.07
+0.5 cm⁻¹     -1.19   1.89  -0.02  0.06            -0.03  0.05
+1 cm⁻¹       -1.88   3.35  -0.04  0.13            -0.06  0.09
+2 cm⁻¹       -1.96   5.24  -0.08  0.24            -0.11  0.17
+4 cm⁻¹        1.77   7.66  -0.02  0.36            -0.18  0.27

Clay reacts most. Moving the bands 1 cm⁻¹ lower changes the clay predictions by 5.4 percentage points RMS, more than the validation RMSE of 4.7. Moving them 2 cm⁻¹ lower changes clay by 13.2 points RMS, with a bias of +10.9. The response is not symmetric: moving the bands higher by the same amount changes clay by less, with a bias of about -2 points. At 2 cm⁻¹ in either direction, pH changes by 0.21 to 0.24 RMS, against a validation RMSE of 0.35, and K by 0.17 to 0.23, against 0.16.

## Wavenumber stretch

A stretch moves each band in proportion to its wavenumber, as an error in the laser wavenumber of an FTIR instrument does. A stretch of 0.001 moves a band at 4000 cm⁻¹ by 4 cm⁻¹, and a band at 1000 cm⁻¹ by 1 cm⁻¹:

In [ ]:
#| rtdata
sweep({f'{s:+g}': partial(perturb, stretch=s) for s in [-1e-3, -5e-4, -2.5e-4, 2.5e-4, 5e-4, 1e-3]}).round(2)

clay_perc       pH_H20       potassium_cmolkg      
              bias   rms   bias   rms             bias   rms
-0.001        3.56  5.66   0.09  0.20             0.15  0.19
-0.0005       1.20  2.73   0.06  0.12             0.10  0.13
-0.00025      0.55  1.43   0.03  0.06             0.06  0.07
+0.00025     -0.43  1.50  -0.02  0.07            -0.03  0.05
+0.0005      -0.73  2.79  -0.03  0.13            -0.07  0.10
+0.001       -1.29  4.81  -0.04  0.24            -0.10  0.15

A stretch of 0.001 moves the bands by 0.65 cm⁻¹ at 650 cm⁻¹ and by 4 cm⁻¹ at 4000 cm⁻¹. It changes clay by 4.8 to 5.7 points RMS, about as much as a uniform shift of 1 cm⁻¹.

## Broadening

A coarser resolution widens every band. KSSL records at 4 cm⁻¹, and OSU at 8 cm⁻¹. To give a resolution R, the filter has σ = √(R² − 4²) / 2.355, which treats both resolutions as the FWHM of a Gaussian line. 16 cm⁻¹ is a coarse extreme:

In [ ]:
#| rtdata
sigma = lambda res: np.sqrt(res**2 - 4**2)/2.355
sweep({f'{r} cm⁻¹': partial(perturb, sigma_cm=sigma(r)) for r in [6, 8, 16]}).round(2)

clay_perc       pH_H20       potassium_cmolkg      
             bias   rms   bias   rms             bias   rms
6 cm⁻¹       0.05  1.31  -0.01  0.05             0.03  0.05
8 cm⁻¹      -0.11  2.44  -0.02  0.09             0.05  0.08
16 cm⁻¹     -0.35  6.11  -0.08  0.32             0.18  0.24

Broadening changes the predictions with almost no bias. Broadening to 8 cm⁻¹, the resolution of OSU, changes clay by 2.4 points RMS, half its validation RMSE. Broadening to 16 cm⁻¹ changes clay by 6.1 points and K by 0.24, more than their validation RMSE.

## Contrast

Contrast raises the absorbance to a power γ. The values fitted in `03_instrument_model` are 0.59 for IAEA and 1.34 for OSU, so the cases go from 0.6 to 1.4:

In [ ]:
#| rtdata
sweep({f'γ {g}': partial(perturb, gamma=g) for g in [0.6, 0.8, 0.9, 1.1, 1.2, 1.4]}).round(2)

clay_perc       pH_H20       potassium_cmolkg      
           bias   rms   bias   rms             bias   rms
γ 0.6      1.29  2.88   0.12  0.24             0.01  0.08
γ 0.8      0.51  1.37   0.06  0.12             0.01  0.05
γ 0.9      0.22  0.71   0.03  0.06             0.01  0.03
γ 1.1     -0.17  0.66  -0.03  0.06            -0.01  0.03
γ 1.2     -0.31  1.22  -0.06  0.11            -0.01  0.05
γ 1.4     -0.52  2.35  -0.11  0.21            -0.03  0.09

At the fitted values of IAEA and OSU, γ = 0.6 and 1.4, contrast changes clay by 2.9 and 2.4 points RMS.

## Gain and baseline

Gain multiplies the spectrum by a smooth curve, and baseline adds one. Both are B-splines with 10 basis functions, as in `InstrumentModel`. Their coefficients are drawn from a normal distribution with standard deviation `sd`, around 1 for gain and around 0 for baseline. Baseline coefficients are in absorbance, and the median absorbance of the OSSL spectra is about 0.9. Each `sd` has five draws. The bias of one draw has an arbitrary sign, so the table gives the mean over draws of the absolute bias and of the RMS change:

In [ ]:
#| rtdata
draws = lambda sd, seed: [np.random.default_rng(seed + d).normal(0, sd, 10) for d in range(5)]
gb = {('gain', sd, d): partial(perturb, gain=c) for sd in [0.01, 0.03, 0.1] for d, c in enumerate(draws(sd, 0))}
gb |= {('baseline', sd, d): partial(perturb, baseline=c) for sd in [0.003, 0.01, 0.03, 0.1] for d, c in enumerate(draws(sd, 100))}
sweep(gb).abs().groupby(level=[0, 1]).mean().round(2)

clay_perc       pH_H20       potassium_cmolkg      
                    bias   rms   bias   rms             bias   rms
baseline 0.003      0.06  0.12   0.00  0.01             0.00  0.00
         0.010      0.20  0.39   0.01  0.04             0.00  0.01
         0.030      0.60  1.15   0.04  0.11             0.01  0.03
         0.100      1.92  3.69   0.19  0.36             0.04  0.10
gain     0.010      0.32  0.48   0.02  0.03             0.01  0.02
         0.030      0.99  1.44   0.06  0.10             0.02  0.05
         0.100      3.73  4.94   0.16  0.31             0.09  0.15

For their size, smooth gain and baseline curves change the predictions least. A baseline with `sd` 0.003 absorbance, the RMS size of D, changes clay by 0.12 points RMS, against 2.8 for D itself in the next section. Only a gain with `sd` 0.1, a variation of about 10% along the spectrum, changes clay by about its validation RMSE (4.9 points).

## Measured differences

D is the mean difference between the KSSL ring-trial spectra and the OSSL spectra of the ring-trial soils that are in OSSL. The fitted models are those of `03_instrument_model`, fitted here for all 20 instruments on the soils that each instrument and KSSL both measured. Each turns a KSSL spectrum into a spectrum of that instrument. Applied to OSSL spectra, the models of MSU, IAEA and OSU show how the CNNs would react if these instruments had measured the OSSL soils. The bias here has a sign:

In [ ]:
#| rtdata
rt = load_ringtrial(rt_path())
mt = ossl_matches(rt['KSSL'], om)
mt = mt[mt.ossl_id.isin(ossl_overlap)]
D = (rt['KSSL'].spectra[pd.Index(rt['KSSL'].sample_ids).get_indexer(mt.index)]
     - om.spectra[pd.Index(om.sample_ids).get_indexer(mt.ossl_id)]).mean(0)
others = [k for k in rt if k != 'KSSL']
effects = {'gain+baseline': ('gain', 'baseline'), 'all five': ('shift', 'broaden', 'contrast', 'gain', 'baseline')}
paired = {i: paired_spectra(rt, i) for i in others}
rt_fits = {(i, n): fit_instrument(wn, paired[i][2], paired[i][1], e) for i in others for n, e in effects.items()}
apply = lambda m: lambda X: m(torch.tensor(X, dtype=torch.float64)).detach().numpy()
sweep({('D', ''): lambda X: X + D} | {k: apply(m) for k, m in rt_fits.items() if k[0] in ['MSU', 'IAEA_aug2022', 'OSU']}).round(2)

clay_perc             ... potassium_cmolkg      
                                bias        rms  ...             bias   rms
D                               2.31   2.780000  ...             0.00  0.04
IAEA_aug2022 gain+baseline     -7.63  11.060000  ...            -0.00  0.28
             all five          -7.21  11.820000  ...            -0.06  0.30
MSU          gain+baseline      1.51   2.230000  ...             0.06  0.08
             all five           0.45   1.880000  ...             0.05  0.07
OSU          gain+baseline      7.41   8.760000  ...             0.28  0.33
             all five          17.17  18.559999  ...             0.32  0.38

[7 rows x 6 columns]

D changes clay by +2.3 points on average. A smooth baseline of the same RMS size changes clay by about 0.1 points, so the model reacts to the narrow bands where D lies, not to its size. The fitted effects of IAEA and OSU change clay by 7 to 17 points, and those of MSU by 2 points or less. For OSU, adding shift, broadening and contrast to gain and baseline doubles the clay bias, from +7.4 to +17.2 points, although in `03_instrument_model` these three effects barely improved the fit to the spectra.

The fitted models reproduce only the part of an instrument's difference from KSSL that all its soils share. On the ring-trial soils, the change in prediction from the KSSL spectra to the instrument's measured spectra (`measured`) can be compared with the change from the KSSL spectra to the output of each fitted model. The table gives the bias of each change, for every instrument:

In [ ]:
#| rtdata
def ring_trial_change(inst):
    ids, Xi, Xr = paired[inst]
    cases = {'measured': lambda X: Xi} | {n: apply(rt_fits[(inst, n)]) for n in effects}
    return pd.concat({p: sensitivity(models[p], Xr, cases).bias for p in ossl_props}, axis=1).unstack()
changes = pd.DataFrame({i: ring_trial_change(i) for i in others}).T
changes.round(2)

clay_perc                ...                       
                  measured gain+baseline  ... gain+baseline all five
Agrocares        -3.790000          4.59  ...          0.18     0.08
Argonne          -3.180000          5.85  ...          0.13     0.06
CSU_IL           -0.330000          1.11  ...          0.10     0.13
ETH_alpha_1      -5.480000          3.77  ...          0.32     0.34
ETH_alpha_2      -3.330000          5.73  ...          0.32     0.34
ETH_vertex        0.310000          4.50  ...          0.17     0.21
IAEA_aug2022    -22.120001        -11.75  ...          0.02    -0.04
LandCare          1.630000          0.92  ...          0.04     0.07
Lesotho          -6.690000         -2.72  ...          0.08     0.26
MSU               0.280000          1.02  ...          0.06     0.05
OSU              -2.400000          8.07  ...          0.29     0.32
Rothamsted       -0.860000          2.20  ...          0.15     0.21
Scion             5.560000          6.08  ...          0.15     0.16
UGhent            0.920000          1.65  ...          0.01     0.01
UIUC             -3.100000          3.09  ...         -0.09    -0.18
USP              -1.590000          3.18  ...          0.25     0.22
UWisc_fine        0.610000          1.22  ...         -0.09    -0.06
Woodwell_alpha   -2.650000          5.37  ...          0.19     0.16
Woodwell_vertex  -2.990000          7.52  ...          0.36     0.35
Hokudai          -5.820000          0.33  ...          0.25     0.30

[20 rows x 9 columns]

The shared effects do not reproduce the measured change in clay. They raise the clay predictions on 18 of the 20 instruments, with a median of +3.1 points for gain and baseline, while the measured spectra lower them on 14, with a median of -2.5 points. Across instruments, the two changes correlate at r = 0.70: the instruments with the largest measured change also have the largest modelled change, but the modelled change is higher. IAEA has the largest measured change, -22 points, and the shared effects give about half of it. For K, the measured spectra raise the predictions on 15 instruments, and gain and baseline on 18, with medians of +0.10 and +0.15 in `log1p(K)`. The shared effects give the direction of the K change on most instruments, but not which instruments have the largest (r = 0.21).

## A soil-specific part shared by the labs

In `03_instrument_model`, the first component of the soil-specific residuals lies on the quartz bands, and its scores correlate with clay with the same sign in 14 of the 20 labs. Part of the soil-specific difference may therefore be common to the labs: a difference between each soil's KSSL spectrum and its spectrum in any other lab. For each instrument, the mean SNV residual of each soil over the other 19 instruments estimates this part without using the instrument's own residuals. It is added to the SNV of the gain and baseline output. The table gives, over the 20 instruments, the correlation of each modelled change with the measured change, and their mean absolute difference:

In [ ]:
#| rtdata
snv = lambda X: (X - X.mean(1, keepdims=True))/X.std(1, keepdims=True)
gb_out = lambda i: apply(rt_fits[(i, 'gain+baseline')])(paired[i][2])
resid = {i: pd.DataFrame(snv(paired[i][1]) - snv(gb_out(i)), index=paired[i][0]) for i in others}
def common_change(inst):
    ids, Xi, Xr = paired[inst]; Xm = gb_out(inst)
    rbar = pd.concat([resid[k].reindex(ids) for k in others if k != inst]).groupby(level=0).mean().reindex(ids).values
    cases = {'measured': lambda X: Xi, 'gain+baseline': lambda X: Xm, '+ common residual': lambda X: snv(Xm) + rbar}
    return pd.concat({p: sensitivity(models[p], Xr, cases).bias for p in ossl_props}, axis=1).unstack()
common = pd.DataFrame({i: common_change(i) for i in others}).T
pd.DataFrame({(p, c): {'r with measured': common[(p, c)].corr(common[(p, 'measured')]),
                       'mean |gap|': (common[(p, c)] - common[(p, 'measured')]).abs().mean()}
              for p in ossl_props for c in ['gain+baseline', '+ common residual']}).round(2)

clay_perc  ...                  
                gain+baseline  ... + common residual
r with measured          0.70  ...              0.25
mean |gap|               5.41  ...              0.10

[2 rows x 6 columns]

Adding the common residual brings the modelled change closer to the measured change. The mean absolute difference falls from 5.4 to 3.6 points for clay, from 0.23 to 0.13 for pH, and from 0.13 to 0.10 for K. The rest of the soil-specific difference belongs to each lab.

## Consequences for augmentation

- The models react most to a wavenumber shift or stretch, at sizes that real instruments have. An augmentation engine needs both, with shifts of at least ±2 cm⁻¹, the size fitted for IAEA.
- Broadening to 8 cm⁻¹ and contrast within the fitted range change the predictions by about half the validation RMSE.
- Smooth gain and baseline curves change the predictions least for their size.
- The models react strongly to narrow features such as those of D, which smooth gain and baseline cannot produce.
- On the ring-trial soils, the shared effects fitted for each instrument raise the clay predictions on 18 of the 20 instruments, while the measured spectra lower them on 14. For K, both raise the predictions on most instruments.
- A soil-specific difference common to the labs explains about a third of the clay gap. Augmentation can add such terms at random for each scan, such as the quartz-band and water components of `03_instrument_model`.
- This check does not show whether augmentation makes the models stable on the measured spectra. Pretraining with augmentation, then scoring on the ring trial, is the test of that.